In [1]:
import os 

os.environ["CUDA_VISIBLE_DEVICES"]="1"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

from transformers import AutoConfig, FlaxAutoModel, AutoTokenizer, FlaxBertPreTrainedModel
from transformers.models.bert.modeling_flax_bert import FlaxBertModule
import jax
from jax import numpy as jnp
from flax import linen as nn
from flax.struct import PyTreeNode
from flax import nnx
from flax.nnx import bridge
import flax
import optax
from typing import Sequence

In [2]:
class Foo(nn.Module):
  
  @nn.compact
  def __call__(self, x):
    return nn.Sequential([nn.Dense(4),
                          nn.Dropout(rate=0.1, deterministic=False),
                          nn.relu,
                          nn.Dense(2),
                          nn.Dropout(rate=0.1, deterministic=False),
                          nn.Dense(2)])(x)

In [3]:
q_model = nnx.bridge.ToNNX(Foo(), rngs=nnx.Rngs(0, dropout=0)).lazy_init(jnp.ones((1, 10)))
q_model.train() # set deterministic=False


class TrainState(nnx.TrainState):
    other_variables: nnx.State

graph, model_params, model_stats = nnx.split(q_model, nnx.Param, ...)
state = TrainState.create(graph, params=model_params, other_variables=model_stats, tx=optax.adam(1e-3))

In [6]:
@jax.jit
def train_step(state: TrainState, x):

  model = nnx.merge(state.graphdef, state.params, state.other_variables)

  def loss_fn(m):
    res1 = m(x)
    return (res1 ** 2).mean()
  
  loss, grads = nnx.value_and_grad(loss_fn)(model)
  _, _, model_stats = nnx.split(model, nnx.Param, ...)

  state = state.apply_gradients(grads=grads, other_variables=model_stats)

  return state, loss

key = jax.random.PRNGKey(0)
jax_train = jax.jit(train_step).lower(state, jax.random.normal(key, (64, 10))).compile()

In [7]:
from tqdm.notebook import tqdm
key = jax.random.PRNGKey(0)

for i in tqdm(range(50000)):
    key, keyi = jax.random.split(key)
    state, loss = jax_train(state, jax.random.normal(keyi, (64, 10)))
    if i % 100 == 0:
        print(state.step, loss)

  0%|          | 0/50000 [00:00<?, ?it/s]

27075 8.636399e-10
27175 4.229938e-12
27275 3.649672e-13
27375 4.7669246e-09
27475 1.2676962e-10
27575 3.1987572e-13
27675 9.484861e-16
27775 4.807841e-17
27875 2.154114e-18
27975 5.5003806e-09
28075 1.2418402e-12
28175 1.1235941e-15
28275 2.8082223e-16
28375 1.4872759e-18
28475 3.627413e-20
28575 8.253802e-09
28675 7.365855e-12
28775 4.3572085e-13
28875 1.04610914e-13
28975 6.82262e-11
29075 8.4160695e-10
29175 1.5805506e-09
29275 1.0504558e-10
29375 2.0589853e-11
29475 2.2037062e-12
29575 2.1144025e-10
29675 1.2595523e-10
29775 3.506566e-11
29875 1.4816008e-10
29975 3.3301212e-10
30075 1.983993e-09
30175 4.2366537e-11
30275 7.332494e-11
30375 9.631325e-10
30475 3.7231086e-11
30575 5.92917e-12
30675 1.9939522e-10
30775 4.901001e-11
30875 5.5383714e-10
30975 7.634051e-10
31075 3.5954895e-11
31175 1.10582724e-10
31275 1.3249277e-11
31375 3.462286e-10
31475 3.3100875e-09
31575 9.1777363e-10
31675 5.3120397e-10
31775 5.2475656e-11
31875 1.1292347e-10
31975 8.7387123e-13
32075 4.0077518e-1

KeyboardInterrupt: 